# AD9226 AXI DMA Capture Demo

DMA version: `AD9226 -> RTL AXIS -> AXIS Data FIFO -> AXI DMA S2MM -> DDR`. Use IP-local offsets with `ip.write(offset, value)`.

In [ ]:
from time import time
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

overlay = Overlay('base_add.bit')
print('IP names:', list(overlay.ip_dict.keys()))

def find_overlay_attr(text):
    matches = [name for name in overlay.ip_dict.keys() if text in name.lower()]
    if not matches:
        raise RuntimeError('Cannot find IP containing %r' % text)
    name = matches[0]
    if '/' in name:
        raise RuntimeError('Hierarchical IP name %r needs manual binding' % name)
    return name, getattr(overlay, name)

ctrl_name, ctrl = find_overlay_attr('adc_capture')
dma_name, dma = find_overlay_attr('dma')
print('Capture IP:', ctrl_name, hex(overlay.ip_dict[ctrl_name]['phys_addr']))
print('DMA IP    :', dma_name, hex(overlay.ip_dict[dma_name]['phys_addr']))

In [ ]:
CTRL = 0x00
STATUS = 0x04
SAMPLE_COUNT = 0x08
ADC_HALF = 0x0C
SAMPLE_DELAY = 0x10
DECIMATION = 0x14
CHANNEL_MASK = 0x18
CAPTURE_MODE = 0x1C
TRIGGER_MODE = 0x20
PRE_DELAY = 0x24
BUFFER_SELECT = 0x28
LATEST_A = 0x2C
LATEST_B = 0x30
SAMPLE_COUNTER = 0x34
FIFO_LEVEL = 0x38
ERROR_FLAGS = 0x3C
LED_CTRL = 0x40
VERSION = 0x44
SAVED_COUNTER = 0x48
LAST_AXIS_WORD = 0x4C
DEBUG_STATE = 0x50
AXIS_SENT_COUNT = 0x54
AXIS_STALL_COUNT = 0x58
TLAST_COUNT = 0x5C
FIFO_BACKPRESSURE = 0x60
DROPPED_SAMPLE_COUNT = 0x64
CAPTURE_DONE_LATCHED = 0x68
CORE_DONE = 0x6C

S2MM_DMASR = 0x34
PL_CLK_HZ = 31_250_000
MAX_SAMPLE_N = 65536
SENTINEL = np.uint32(0xDEADBEEF)

def dma_status():
    s = dma.mmio.read(S2MM_DMASR)
    print('S2MM_DMASR      = 0x%08X' % s)
    print('halted/idle     =', s & 1, (s >> 1) & 1)
    print('int/slv/dec err =', (s >> 4) & 1, (s >> 5) & 1, (s >> 6) & 1)
    print('ioc/err irq     =', (s >> 12) & 1, (s >> 14) & 1)

def dump_regs():
    status = ctrl.read(STATUS)
    print('STATUS          = 0x%08X' % status)
    print('  busy          =', (status >> 0) & 1)
    print('  axis_done     =', (status >> 1) & 1)
    print('  fatal_error   =', (status >> 10) & 1)
    print('ERROR_FLAGS     = 0x%08X  (fatal + warning/debug)' % ctrl.read(ERROR_FLAGS))
    print('LATEST_A/B      = %d / %d' % (ctrl.read(LATEST_A), ctrl.read(LATEST_B)))
    print('SAMPLE_COUNTER  =', ctrl.read(SAMPLE_COUNTER))
    print('FIFO_LEVEL      =', ctrl.read(FIFO_LEVEL))
    print('SAVED_COUNTER   =', ctrl.read(SAVED_COUNTER), '(capture_core attempted count, not DMA success)')
    print('LAST_AXIS_WORD  = 0x%08X' % ctrl.read(LAST_AXIS_WORD))
    print('DEBUG_STATE     =', ctrl.read(DEBUG_STATE))
    print('AXIS_SENT_COUNT =', ctrl.read(AXIS_SENT_COUNT))
    print('AXIS_STALL_COUNT=', ctrl.read(AXIS_STALL_COUNT))
    print('TLAST_COUNT     =', ctrl.read(TLAST_COUNT))
    print('FIFO_BACKPRESS  =', ctrl.read(FIFO_BACKPRESSURE))
    print('DROPPED_SAMPLES =', ctrl.read(DROPPED_SAMPLE_COUNT))
    print('CAPTURE_DONE    =', ctrl.read(CAPTURE_DONE_LATCHED))
    print('CORE_DONE       =', ctrl.read(CORE_DONE))
    print('VERSION         = 0x%08X' % ctrl.read(VERSION))
    dma_status()

dump_regs()

In [ ]:
sample_count = 1024  # quick test; later use 16384 / 32768 / 65536
sample_count = min(max(int(sample_count), 1), MAX_SAMPLE_N)
adc_half_period = 12
decimation = 1
sample_delay = 1
channel_mask = 0b11
capture_mode = 2  # DMA only: 2 fake stream first; 1 real AD9226 after DMA path passes
if capture_mode == 0:
    raise ValueError('capture_mode=0 is legacy HLS fake and will hang AXI DMA; use 2 or 1')
trigger_mode = 0
pre_delay = 0

actual_adc_fs = PL_CLK_HZ / (2 * adc_half_period)
print('sample_count =', sample_count)
print('adc_half_period =', adc_half_period)
print('actual_adc_fs = %.3f MSPS' % (actual_adc_fs / 1e6))

In [ ]:
buf = allocate(shape=(sample_count,), dtype=np.uint32)
buf[:] = SENTINEL
buf.flush()

# DMA must start before capture.
dma.recvchannel.transfer(buf)

# Clear RTL status/FIFO and configure capture.
ctrl.write(CTRL, 0x04)
ctrl.write(CTRL, 0x00)
ctrl.write(ERROR_FLAGS, 0xFFFFFFFF)
ctrl.write(SAMPLE_COUNT, sample_count)
ctrl.write(ADC_HALF, adc_half_period)
ctrl.write(SAMPLE_DELAY, sample_delay)
ctrl.write(DECIMATION, decimation)
ctrl.write(CHANNEL_MASK, channel_mask)
ctrl.write(CAPTURE_MODE, capture_mode)
ctrl.write(TRIGGER_MODE, trigger_mode)
ctrl.write(PRE_DELAY, pre_delay)
ctrl.write(BUFFER_SELECT, 0)

ctrl.write(CTRL, 0x01)
ctrl.write(CTRL, 0x03)
ctrl.write(CTRL, 0x01)

t0 = time()
dma.recvchannel.wait()
elapsed = time() - t0
buf.invalidate()
print('DMA wait elapsed %.6f s' % elapsed)
dump_regs()

In [ ]:
raw = np.array(buf, dtype=np.uint32)
ch0 = raw & np.uint32(0x0FFF)
ch1 = (raw >> np.uint32(16)) & np.uint32(0x0FFF)

if np.any(raw == SENTINEL):
    raise RuntimeError('Sentinel remains in DMA buffer')
if ctrl.read(AXIS_SENT_COUNT) != sample_count:
    raise RuntimeError('AXIS_SENT_COUNT mismatch')
if ctrl.read(TLAST_COUNT) != 1:
    raise RuntimeError('TLAST_COUNT mismatch')
if ctrl.read(DROPPED_SAMPLE_COUNT) != 0:
    raise RuntimeError('Dropped samples detected')
if ctrl.read(STATUS) & (1 << 10):
    raise RuntimeError('STATUS.fatal_error is set; check dropped/config/DMA status')

if capture_mode == 2:
    expected = np.arange(sample_count, dtype=np.uint32) & np.uint32(0x0FFF)
    assert np.array_equal(ch0, expected)
    assert np.array_equal(ch1, np.uint32(4095) - expected)

print('CH0 first 16:', ch0[:16].tolist())
print('CH1 first 16:', ch1[:16].tolist())
print('CH0 mean/Vpp:', float(ch0.mean()), int(ch0.max() - ch0.min()))
print('CH1 mean/Vpp:', float(ch1.mean()), int(ch1.max() - ch1.min()))

nplot = min(sample_count, 512)
plt.figure(figsize=(10, 4))
plt.plot(ch0[:nplot], label='CH0 raw')
plt.plot(ch1[:nplot], label='CH1 raw')
plt.grid(True)
plt.legend()
plt.show()